# An Intelligent Noise-Free Emergency Vehicle Priority Framework Using LoRa Communication and Edge Al Analytics

This project implements a complete Cloud AI inference pipeline and FastAPI edge server designed to prioritize emergency vehicles by intelligently detecting front-vehicle traffic flow without the need for traditional sirens (Noise-Free).

##  System Overview

The system receives telemetry and dual-camera image feeds from an edge device (Raspberry Pi) and applies state-of-the-art Computer Vision algorithms on the server/cloud side:

1. **Traffic Analysis Module (Wide View)**
   * Uses pretrained YOLOv8 (COCO dataset) to dynamically monitor the entire front road scene.
   * Assesses total traffic density and spots congestion.
   * Measures 'free space' directly in front of the target vehicle to see if yielding is physically possible.

2. **OCR & Number Plate Tracking Module (Focused View)**
   * Locates the precise bounding box of the front vehicle's license plate using YOLOv8.
   * Employs real-time image enhancements (Gaussian blur, adaptive thresholding) to clean the cropped patch.
   * Extracts alphanumeric values using EasyOCR for logging.

##  Decision Engine Logic 

The autonomous `PipelineController` executes the following enforcement checks:
* **No action required (Discard Case)**: If heavy traffic congestion restricts movement, OR there is no free space for the front vehicle to move out of the way.
* **Auto-Challan Trigger**: If the AI calculates that the path ahead is clear, but the vehicle intentionally fails to yield to the emergency vehicle. The OCR module saves the license details and logs the event directly into the JSON database.

*Architected for Edge-Cloud Integration and Real-time Processing.*

In [1]:
import os
import cv2
import json
import logging
from datetime import datetime
import numpy as np
import easyocr
import uvicorn
import nest_asyncio
from fastapi import FastAPI, UploadFile, File
from ultralytics import YOLO

# ----------------- LOGGING & STORAGE SETUP -----------------
# Create essential directories for the pipeline
STORAGE_DIR = "logs/pipeline_data"
# Store raw images, crops, metadata
os.makedirs(os.path.join(STORAGE_DIR, "images/wide"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "images/focused"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "images/plates"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "decisions"), exist_ok=True)

# Simple logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("TrafficSystem")

logger.info("Environment and directories initialized.")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\dnyan\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


2026-04-09 23:00:32,798 - INFO - Environment and directories initialized.


In [2]:
# ---------------------------------------------------------
# Training Code: Custom YOLOv8 Number Plate Detection Model
# ---------------------------------------------------------

def train_number_plate_model(data_yaml_path="dataset/data.yaml", epochs=50, imgsz=640, batch=16):
    """
    Train a custom YOLOv8 Object Detection Model to strictly detect number plates
    from focused edge-device images.

    Args:
    - data_yaml_path: Path to your YOLO dataset configuration file specifying train/val splits and classes.
    - epochs: Number of epochs to train for
    - imgsz: Target image size
    - batch: Batch size for training
    """
    logger.info("Initializing YOLOv8n custom training ...")
    
    # Load YOLOv8 Nano base model
    model = YOLO("yolov8n.pt")
    
    # Train the model with the dataset
    results = model.train(
        data=data_yaml_path,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        name="custom_number_plate_model"
    )
    logger.info("Training complete. Model weights saved in 'runs/detect/custom_number_plate_model/weights/best.pt'")
    
    return results

# Uncomment the line below when your dataset is ready and the environment has a GPU available.
# train_number_plate_model()

In [3]:
# ---------------------------------------------------------
# Module 1: Traffic Analysis Model (Wide Image) 
# ---------------------------------------------------------
class TrafficAnalysisModule:
    """
    Analyzes the wide image capturing the full traffic scene ahead of the EV.
    Uses pretrained YOLOv8 on COCO for detecting vehicles (car, truck, bus, bike).
    Estimates density and free space using bounding boxes.
    """
    def __init__(self, model_version="yolov8s.pt"):
        # Load the pretrained model (COCO classes). We'll map indices:
        # 2: car, 3: motorcycle, 5: bus, 7: truck
        logger.info(f"Loading generic traffic model: {model_version}")
        self.model = YOLO(model_version)
        self.target_classes = [2, 3, 5, 7] 

    def analyze_traffic(self, image_np):
        """
        Evaluate traffic state: Detect count, density, and front-vehicle free space.
        Args:
            image_np: Wide frame as NumPy array (H_img, W_img, 3).
        Returns:
            dict: { congestion_detected (bool), free_space (int), objects (list) }
        """
        results = self.model(image_np)
        
        # Parse inference results
        boxes = results[0].boxes
        height, width = image_np.shape[:2]
        total_area = height * width

        detected_vehicles = []
        for box in boxes:
            cls_id = int(box.cls[0].item())
            if cls_id in self.target_classes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                # store center Y point (to figure out depth/proximity)
                cy = (y1 + y2) // 2
                detected_vehicles.append({
                    "class_id": cls_id,
                    "bbox": [x1, y1, x2, y2],
                    "area": (x2 - x1) * (y2 - y1),
                    "cy": cy
                })

        # Calculate logical parameters
        num_vehicles = len(detected_vehicles)
        
        # 1. Congestion Check (Density Estimation)
        # Using a simplistic heuristic: if the total area of vehicles takes up >30% of frame 
        # or pure count > high threshold -> Congestion TRUE
        max_vehicles = 10
        total_vehicle_area = sum(v["area"] for v in detected_vehicles)
        density_ratio = total_vehicle_area / total_area

        is_congested = (num_vehicles > max_vehicles) or (density_ratio > 0.3)
        
        # 2. Free Space calculation in front of the vehicle
        # Let's say the lowest "cy" (highest block on image, mathematically near bottom of frame)
        # is the frontmost vehicle. We measure distance from bottom of screen to that vehicle.
        # Alternatively, find the gap between the middle vehicle and others.
        free_space_pixels = 0
        if num_vehicles > 0:
            # Sort by cy descending (closest to bottom of image is index 0)
            sorted_by_y = sorted(detected_vehicles, key=lambda v: list(v["bbox"])[3], reverse=True)
            front_vehicle = sorted_by_y[0]
            # Space left in front of the front vehicle is distance between it and the next vehicle upwards
            # If it's the only vehicle, free space is large.
            if len(sorted_by_y) > 1:
                next_vehicle = sorted_by_y[1]
                free_space_pixels = abs(front_vehicle["bbox"][1] - next_vehicle["bbox"][3])
            else:
                # If only 1 car is detected, we assume full free space above it
                free_space_pixels = front_vehicle["bbox"][1]

        logger.info(f"Traffic Analysis | Count: {num_vehicles} | Density: {density_ratio:.2f} | Congested: {is_congested} | Free Space Gap: {free_space_pixels}px")
        
        # Render debugging image
        debug_img = results[0].plot()

        return {
            "vehicle_detected_in_front": num_vehicles > 0,
            "traffic_density_ratio": float(density_ratio),
            "is_congested": is_congested,
            "free_space_pixels": free_space_pixels,
            "debug_image": debug_img
        }

traffic_module = TrafficAnalysisModule()

2026-04-09 23:01:11,475 - INFO - Loading generic traffic model: yolov8s.pt


In [4]:
# ---------------------------------------------------------
# Module 2: OCR and Number Plate Recognition (Focused Image)
# ---------------------------------------------------------

class NumberPlateOCRModule:
    """
    Combines custom trained YOLOv8 Object Detection strictly for 'Number Plate' 
    with a preprocessing pipeline and EasyOCR.
    """
    def __init__(self, custom_weights_path="runs/detect/custom_number_plate_model/weights/best.pt"):
        # Placeholder for custom trained model. If the file doesn't exist (e.g., prototype), 
        # we back fall to a dummy or YOLOv8n for the sake of pipeline demonstration.
        try:
            self.np_detector = YOLO(custom_weights_path)
            logger.info("Loaded custom trained YOLO plate model.")
        except Exception as e:
            logger.warning(f"Custom model '{custom_weights_path}' not found. Falling back to COCO yolov8n.pt for pipeline init logic.")
            self.np_detector = YOLO('yolov8n.pt')

        self.reader = easyocr.Reader(['en'], gpu=True) # Needs GPU. Set to False if running edge CPU only.

    def preprocess_plate(self, plate_img):
        """
        Enhance the cropped number plate image for better text recognition.
        Steps: Grayscale -> Gaussian Blur/Denose -> Adaptive Thresholding.
        """
        # Convert to gray
        gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)
        
        # Blur the image slightly to remove noise
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        
        # Apply adaptive threshold to highlight characters against background
        thresh = cv2.adaptiveThreshold(
            blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
        
        return thresh

    def detect_and_read(self, focused_img):
        """
        1. Uses YOLO to find bounding box coordinates of number plates.
        2. Crops that area.
        3. Preprocesses the plate for OCR.
        4. Extracts text.
        """
        # Run custom YOLO inference
        results = self.np_detector(focused_img)
        boxes = results[0].boxes

        extracted_texts = []
        best_plate_crop = None

        if len(boxes) == 0:
            logger.info("No number plate detected in focused image.")
            return None, None

        # Take the box with highest confidence
        sorted_boxes = sorted(boxes, key=lambda b: b.conf[0].item(), reverse=True)
        best_box = sorted_boxes[0]

        x1, y1, x2, y2 = map(int, best_box.xyxy[0].tolist())
        conf = best_box.conf[0].item()

        # Crop out plate
        best_plate_crop = focused_img[y1:y2, x1:x2].copy()

        # Preprocess plate for Tesseract/EasyOCR
        processed_plate = self.preprocess_plate(best_plate_crop)

        # Apply EasyOCR
        ocr_result = self.reader.readtext(processed_plate)

        # Standardize and filter the result (removing spaces/special chars)
        for bbox, text, score in ocr_result:
            clean_text = "".join(e for e in text if e.isalnum()).upper()
            if len(clean_text) > 4: # Typical plate length filter
                extracted_texts.append({
                    "text": clean_text,
                    "confidence": score,
                    "bbox": bbox
                })

        logger.info(f"YOLO Plate detection Conf: {conf:.2f}. OCR results: {extracted_texts}")
        
        # Sort out best OCR result
        if not extracted_texts:
            return None, best_plate_crop

        best_text = sorted(extracted_texts, key=lambda tb: tb["confidence"], reverse=True)[0]
        return best_text["text"], best_plate_crop

ocr_module = NumberPlateOCRModule()

2026-04-09 23:01:39,850 - WARNING - Custom model 'runs/detect/custom_number_plate_model/weights/best.pt' not found. Falling back to COCO yolov8n.pt for pipeline init logic.


2026-04-09 23:01:43,301 - WARNING - Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
2026-04-09 23:01:43,305 - WARNING - Downloading detection model, please wait. This may take several minutes depending upon your network connection.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

2026-04-09 23:02:30,290 - INFO - Download complete
2026-04-09 23:02:30,290 - WARNING - Downloading recognition model, please wait. This may take several minutes depending upon your network connection.


Progress: |█████████████████████████████████████████████████-| 98.5% Complete

2026-04-09 23:02:38,966 - INFO - Download complete.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [5]:
# ---------------------------------------------------------
# Module 3: Decision Logic & Priority System Pipeline
# ---------------------------------------------------------

class PipelineController:
    """
    Coordinates outputs between Wide and Focused image processing.
    Decides when to trigger OCR, saving logs, and enforcing CHallan logic.
    """
    def __init__(self, traffic_module, ocr_module):
        self.traffic_module = traffic_module
        self.ocr_module = ocr_module
        
        # Hyperparameters
        self.TRAFFIC_DENSITY_THRESHOLD = 0.3    # ~30% area filled
        self.FREE_SPACE_PIX_MINIMUM = 50        # minimum 50 pixels threshold

    def evaluate_priority(self, wide_img, focused_img, rssi_placeholder=None):
        """
        Input: the two images from edge device
        Output: Dictionary specifying whether a Challan logic applies
        """
        logger.info(f"evaluating priority logic...")

        # Step 1: Wide Image (Traffic Analysis)
        traffic_data = self.traffic_module.analyze_traffic(wide_img)

        # Variables for readability
        vehicle_detected_in_front = traffic_data["vehicle_detected_in_front"]
        density = traffic_data["traffic_density_ratio"]
        free_space = traffic_data["free_space_pixels"]
        is_congested = traffic_data["is_congested"]

        decision_status = ""
        action_taken = "Ignore"
        extracted_plate = None
        
        # Step 2: Decision Logic Check
        """
        IF (vehicle_detected_in_front == TRUE)
           AND (traffic_density < threshold)
           AND (free_space > threshold)
        THEN
           perform_number_plate_detection()
           generate_challan()
        ELSE
           ignore_case()
        """
        if is_congested:
           decision_status = "Congestion detected. Ignoring case." 

        elif not vehicle_detected_in_front:
           decision_status = "No vehicles detected in front. Ignoring case."
           
        elif free_space < self.FREE_SPACE_PIX_MINIMUM:
           decision_status = "Not enough free space to move out of EV's way. Ignoring case."

        else:
           # Conditions Met
           decision_status = f"Space available ({free_space}px). Initiating automated system."
           action_taken = "Proceeding to OCR"
           
           # Trigger Focused OCR Analysis
           plate_text, cropped_plate_img = self.ocr_module.detect_and_read(focused_img)
           if plate_text:
                action_taken = "Challan Generated"
                extracted_plate = plate_text
                # Log successfully cropped plate
                filename = f"logs/pipeline_data/plates/{plate_text}_{datetime.now().strftime('%H%M%S')}.jpg"
                cv2.imwrite(filename, cropped_plate_img)
           else:
                action_taken = "Error Processing OCR"
           
        payload = {
            "timestamp": str(datetime.now()),
            "status": decision_status,
            "action": action_taken,
            "metrics": {
                "density": f"{density:.2f}",
                "free_space_pixels": free_space,
                "rssi_value": rssi_placeholder
            },
            "plate_number": extracted_plate
        }

        # Backup Pipeline Storage
        log_json_path = f"logs/pipeline_data/decisions/decision_{datetime.now().strftime('%Y%m%d%H%M%S')}.json"
        with open(log_json_path, 'w') as f:
            json.dump(payload, f, indent=4)
        
        return payload

pipeline = PipelineController(traffic_module, ocr_module)

In [6]:
# ---------------------------------------------------------
# Web API Module: FastAPI Server (Cloud Receiver)
# ---------------------------------------------------------

app = FastAPI(title="Emergency Vehicle Priority AI Backend")

@app.post("/process-event")
async def process_traffic_event(wide_img: UploadFile = File(...), focused_img: UploadFile = File(...), rssi: int = None):
    """
    Endpoint mapping Raspberry Pi data transfers to the AI pipeline.
    Expects two images (wide and focused view) and an optional RSSI value.
    """
    logger.info("Received event from Edge Device.")
    
    # Read files into memory
    wide_bytes = await wide_img.read()
    focused_bytes = await focused_img.read()

    # Convert binary to OpenCV images
    wide_np = cv2.imdecode(np.frombuffer(wide_bytes, np.uint8), cv2.IMREAD_COLOR)
    focused_np = cv2.imdecode(np.frombuffer(focused_bytes, np.uint8), cv2.IMREAD_COLOR)

    # Save incoming raw files
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    cv2.imwrite(f"logs/pipeline_data/images/wide/wide_{timestamp}.jpg", wide_np)
    cv2.imwrite(f"logs/pipeline_data/images/focused/focused_{timestamp}.jpg", focused_np)

    # Pass to logic controller
    pipeline_result = pipeline.evaluate_priority(wide_np, focused_np, rssi)

    return {"status": "success", "pipeline_evaluation": pipeline_result}

# To run FastAPI inside a Jupyter Notebook context, use nest_asyncio and uvicorn programmatically.
def run_server():
    nest_asyncio.apply()
    logger.info("Starting FastAPI server at http://localhost:8000")
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Uncomment the line below to start the server!
# run_server()

In [ ]:
# ---------------------------------------------------------
# Edge Client Simulator: Raspberry Pi Test Code
# ---------------------------------------------------------

import requests
import time
import os
import io
import numpy as np

def simulate_raspberry_pi_call():
    """
    Mock script to test the Edge-to-Cloud API locally.
    Creates dummy image data (since you need physical cameras later)
    and sends it via a POST request to the running FastAPI server.
    """
    url = "http://127.0.0.1:8000/process-event"

    # Create dummy images representing 'wide' and 'focused' captures
    # 3 channel empty array
    dummy_wide = np.zeros((480, 640, 3), dtype=np.uint8)
    dummy_focused = np.zeros((480, 640, 3), dtype=np.uint8)

    # Encode to bytes as if reading a raw .jpg
    _, wide_enc = cv2.imencode('.jpg', dummy_wide)
    _, focused_enc = cv2.imencode('.jpg', dummy_focused)

    wide_io = io.BytesIO(wide_enc.tobytes())
    focused_io = io.BytesIO(focused_enc.tobytes())

    # Send POST with Files as Multipart form data
    files = {
        'wide_img': ('wide.jpg', wide_io, 'image/jpeg'),
        'focused_img': ('focused.jpg', focused_io, 'image/jpeg')
    }

    print(f"[{time.strftime('%H:%M:%S')}] Edge Node sending telemetry to {url} ...")
    try:
        response = requests.post(url, files=files, params={"rssi": -65})
        print(f"Cloud Response Status: {response.status_code}")
        print(f"Evaluation JSON Result:\n {json.dumps(response.json(), indent=2)}")
    except requests.exceptions.ConnectionError:
        print("Error: Could not connect to FastAPI server. Ensure it is running in another thread or cell!")

# Uncomment below to Test
# simulate_raspberry_pi_call()

In [ ]:
# --- TEST CELL: Verify Pipeline Flow with Dummy Data ---

import numpy as np
import cv2
import json

def test_pipeline():
    logger.info("Starting pipeline test...")
    # Create dummy images (e.g. empty images to simulate camera)
    dummy_wide = np.zeros((480, 640, 3), dtype=np.uint8)
    dummy_focused = np.zeros((480, 640, 3), dtype=np.uint8)

    # Let's run it directly through the core logic
    result = pipeline.evaluate_priority(dummy_wide, dummy_focused, rssi_placeholder=-65)
    
    print("\n--- Pipeline Emulation Output ---")
    print(json.dumps(result, indent=2))
    
test_pipeline()


2026-04-09 23:03:32,292 - INFO - Starting pipeline test...
2026-04-09 23:03:32,293 - INFO - evaluating priority logic...



0: 480x640 (no detections), 164.8ms
Speed: 6.4ms preprocess, 164.8ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)


2026-04-09 23:03:32,554 - INFO - Traffic Analysis | Count: 0 | Density: 0.00 | Congested: False | Free Space Gap: 0px



--- Pipeline Emulation Output ---
{
  "timestamp": "2026-04-09 23:03:32.556060",
  "status": "No vehicles detected in front. Ignoring case.",
  "action": "Ignore",
  "metrics": {
    "density": "0.00",
    "free_space_pixels": 0,
    "rssi_value": -65
  },
  "plate_number": null
}
